In [1]:
# Permite ajustar la anchura de la parte útil de la libreta (reduce los márgenes)
from IPython.core.display import display, HTML
display(HTML("<style>.container{ width:98% }</style>"))

/tmp/ipykernel_1253191/1126351303.py:2: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import display, HTML


In [ ]:
# ## Se requiere instalar estos paquetes
# !pip install swig
# !pip install wrds
# !pip install pyportfolioopt
# ## install finrl library
# !pip install git+https://github.com/AI4Finance-Foundation/FinRL.git

In [2]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from finrl.meta.preprocessor.yahoodownloader import YahooDownloader
from finrl.meta.env_stock_trading.env_stocktrading import StockTradingEnv
from finrl.agents.stablebaselines3.models import DRLAgent
from stable_baselines3 import A2C, DDPG, PPO, SAC, TD3

%matplotlib inline
from finrl.config import INDICATORS

2025-05-20 21:04:00.186497: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-05-20 21:04:00.195606: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-05-20 21:04:00.205396: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-05-20 21:04:00.208153: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-05-20 21:04:00.215377: I tensorflow/core/platform/cpu_feature_guar

# Backtesting

In [3]:
import pickle

with open("train.pkl", "rb") as f:
    train = pickle.load(f)
    
with open("trade.pkl", "rb") as f:
    trade = pickle.load(f)

# train = train.set_index(train.columns[0])
# train.index.names = ['']
# trade = trade.set_index(trade.columns[0])
# trade.index.names = ['']

In [4]:
if_using_a2c = False
if_using_ddpg = False
if_using_ppo = True
if_using_td3 = False
if_using_sac = False

In [5]:
ruta = "./results/ppo/agent_ppo"
#trained_a2c = A2C.load('trained_models/agent_a2c') if if_using_a2c else None
#trained_ddpg = DDPG.load("trained_models/agent_ddpg") if if_using_ddpg else None
trained_ppo = PPO.load(ruta) if if_using_ppo else None
#trained_td3 = TD3.load("trained_models/agent_td3") if if_using_td3 else None
#trained_sac = SAC.load("trained_models/agent_sac") if if_using_sac else None

## Out-of-Sample Performance

In [6]:
stock_dimension = len(train.tic.unique()) # tickers únicos sin repetir

state_space = 1 + 2*stock_dimension + len(INDICATORS)*stock_dimension

print(f"Number of Indicators: {len(INDICATORS)}")

print(f"Stock Dimension: {stock_dimension}")

print(f"state_space = 1 + 2*{stock_dimension} + ({len(INDICATORS)})*{stock_dimension}")

print(f"State Space: {state_space}")

Number of Indicators: 8
Stock Dimension: 8
state_space = 1 + 2*8 + (8)*8
State Space: 81


In [7]:
buy_cost_list = sell_cost_list = [0.001] * stock_dimension
num_stock_shares = [0] * stock_dimension

env_kwargs = {
    "hmax": 100,
    "initial_amount": 1000000,
    "num_stock_shares": num_stock_shares,
    "buy_cost_pct": buy_cost_list,
    "sell_cost_pct": sell_cost_list,
    "state_space": state_space,
    "stock_dim": stock_dimension,
    "tech_indicator_list": INDICATORS,
    "action_space": stock_dimension,
    "reward_scaling": 1e-4
}

In [8]:
e_trade_gym = StockTradingEnv(df = trade, turbulence_threshold = 70,risk_indicator_col='vix', **env_kwargs)
# env_trade, obs_trade = e_trade_gym.get_sb_env()

### 🧪 Evaluación del agente entrenado en el entorno de prueba

Después de entrenar el agente (ya sea PPO, A2C u otro), llega el momento de **evaluar su desempeño**. Para ello se utiliza un entorno de prueba llamado `e_trade_gym`, que contiene datos de mercado que el agente no vio durante el entrenamiento (por ejemplo, un rango de fechas posterior).

---

#### 🧠 ¿Qué se hace en esta etapa?

Se ejecuta la función:

$$DRLAgent.DRL_prediction(model=..., environment=...)$$

La cual hace que el agente interactúe paso a paso con el entorno de prueba. En cada paso:

- El agente recibe un estado del mercado.
- Toma una decisión (comprar, vender o mantener cada acción).
- Se actualiza el valor del portafolio.
- Se registra esa acción y ese valor.

---

### 📦 ¿Qué resultados se obtienen?

Se generan dos objetos principales para cada agente:

- `df_account_value_ppo` / `df_account_value_a2c`:  
  Un DataFrame que muestra la evolución del valor del portafolio a lo largo del tiempo.

- `df_actions_ppo` / `df_actions_a2c`:  
  Un DataFrame con las decisiones de trading del agente (acciones compradas o vendidas por día).

---

### 🎯 ¿Para qué sirve esto?

Esta evaluación te permite:

- Analizar el comportamiento del agente en datos no vistos.
- Graficar el valor del portafolio a lo largo del tiempo.
- Calcular métricas de rendimiento financiero:
  - Retorno total
  - Ratio de Sharpe
  - Máximo drawdown
- Comparar el desempeño contra otras estrategias (por ejemplo, buy-and-hold).


In [9]:
# Si estamos usando A2C, evaluamos el modelo entrenado en el entorno de prueba (e_trade_gym)
# DRL_prediction simula el comportamiento del agente paso a paso en el mercado
# Devuelve dos DataFrames:
#   - df_account_value_a2c: valor del portafolio a lo largo del tiempo
#   - df_actions_a2c: acciones tomadas (comprar, vender o mantener) por el agente en cada paso
df_account_value_a2c, df_actions_a2c = DRLAgent.DRL_prediction(
    model=trained_a2c,
    environment = e_trade_gym) if if_using_a2c else (None, None)

# Lo mismo, pero si estamos usando PPO: evaluamos el modelo PPO en el entorno de prueba
# Devuelve:
#   - df_account_value_ppo: evolución del portafolio con PPO
#   - df_actions_ppo: decisiones de trading tomadas por PPO
df_account_value_ppo, df_actions_ppo = DRLAgent.DRL_prediction(
    model=trained_ppo,
    environment = e_trade_gym) if if_using_ppo else (None, None)

hit end!


# Mean Variance Optimization

MVO will be the baseline to compare our agent performance

## Preparación de Datos para Optimización de Portafolios (MVO)

En esta sección se presentan funciones que convierten datos históricos de precios en estructuras adecuadas para aplicar modelos clásicos de optimización de portafolio, como la **Media-Varianza de Markowitz (Mean-Variance Optimization, MVO)**.

---

### `process_df_for_mvo(df)`

Esta función transforma un `DataFrame` con datos históricos en una **matriz de precios de cierre**, donde:

- Las **filas** representan fechas.
- Las **columnas** corresponden a los distintos **tickers** (acciones).

Este formato es esencial para realizar análisis financieros tradicionales.

---

### `StockReturnsComputing(StockPrice, Rows, Columns)`

A partir de la matriz de precios generada por `process_df_for_mvo`, esta función calcula los **rendimientos diarios porcentuales** de cada activo.

Los rendimientos se calculan como:

$$
\text{retorno}_{t} = \frac{P_t - P_{t-1}}{P_{t-1}}
$$

donde $P_t$ es el precio en el día $t$.

---

### ¿Qué se obtiene?

Al aplicar ambas funciones se generan dos salidas fundamentales:

- **Matriz de precios**: tickers por columna, fechas por fila.
- **Matriz de rendimientos diarios (%)**: base para análisis estadístico.

---

### ¿Para qué sirven?

Estas matrices pueden utilizarse para:

- Calcular medias, varianzas y covarianzas de los activos.
- Construir la **matriz de covarianza** necesaria para modelos MVO.
- Aplicar **restricciones** y preferencias del inversionista.
- Visualizar y analizar la eficiencia del portafolio.
- Comparar estrategias en términos de **riesgo-retorno**.

Este procesamiento es un paso clave para aplicar técnicas clásicas de **optimización cuantitativa de portafolios**.

In [11]:
# Función que convierte un DataFrame largo (con columnas 'date', 'tic', 'close') 
# en una matriz de precios con fechas como índice y tickers como columnas.
def process_df_for_mvo(df):
  # Ordena el DataFrame por fecha y ticker, y se queda solo con las columnas necesarias
  df = df.sort_values(['date','tic'], ignore_index=True)[['date','tic','close']]
  
  # Toma el primer bloque de datos (una fila por ticker) para extraer la lista de símbolos (tic)
  fst = df.iloc[0:stock_dimension, :]
  tic = fst['tic'].tolist()

  # Inicializa un DataFrame vacío con columnas = tickers, que almacenará los precios por día
  mvo = pd.DataFrame()
  for k in range(len(tic)):
    mvo[tic[k]] = 0  # Llena con ceros inicialmente

  # Recorre el DataFrame completo por bloques (cada bloque corresponde a un día completo)
  for i in range(df.shape[0] // stock_dimension):
    n = df.iloc[i * stock_dimension:(i+1) * stock_dimension, :]  # bloque de datos para una fecha
    date = n['date'].iloc[0]  # extrae la fecha correspondiente a ese bloque
    mvo.loc[date] = n['close'].tolist()  # asigna precios de cierre a esa fecha

  # Devuelve un DataFrame con fechas como índice y tickers como columnas
  return mvo


# Función que calcula los rendimientos diarios (porcentuales) de cada acción a partir de la matriz de precios
def StockReturnsComputing(StockPrice, Rows, Columns):
  import numpy as np
  
  # Inicializa una matriz de ceros para los rendimientos
  # Tiene una fila menos porque los rendimientos se calculan entre pares de días
  StockReturn = np.zeros([Rows - 1, Columns])
  
  # Calcula los rendimientos porcentuales por activo
  for j in range(Columns):        # j recorre las columnas (acciones)
    for i in range(Rows - 1):     # i recorre las filas (días)
      StockReturn[i, j] = ((StockPrice[i+1, j] - StockPrice[i, j]) / StockPrice[i, j]) * 100

  # Devuelve la matriz de rendimientos
  return StockReturn


StockData = process_df_for_mvo(train)
TradeData = process_df_for_mvo(trade)

TradeData.to_numpy()


# Convierte el DataFrame de precios en un arreglo NumPy para facilitar los cálculos
arStockPrices = np.asarray(StockData)

# Extrae el número de filas (días) y columnas (activos) del arreglo de precios
[Rows, Cols] = arStockPrices.shape

# Calcula los rendimientos porcentuales diarios usando la función definida previamente
# arReturns tendrá forma (Rows - 1, Cols), es decir, un rendimiento por día y por activo
arReturns = StockReturnsComputing(arStockPrices, Rows, Cols)

# Calcula los rendimientos medios diarios para cada activo (promedio de cada columna)
meanReturns = np.mean(arReturns, axis=0)

# Calcula la matriz de varianzas y covarianzas de los rendimientos
# Esto representa el riesgo y las correlaciones entre activos
covReturns = np.cov(arReturns, rowvar=False)

# Configura la salida de NumPy para mostrar solo 3 decimales y sin notación científica
np.set_printoptions(precision=3, suppress=True)

# Imprime los resultados: los rendimientos medios diarios y la matriz de covarianza
print('Mean returns of assets in k-portfolio 1\n', meanReturns)
print('Variance-Covariance matrix of returns\n', covReturns)

Mean returns of assets in k-portfolio 1
 [0.136 0.158 0.066 0.103 0.033 0.076 0.103 0.038]
Variance-Covariance matrix of returns
 [[3.156 1.592 1.722 1.303 1.218 1.674 1.561 0.876]
 [1.592 4.533 1.657 1.299 1.158 1.516 1.808 0.761]
 [1.722 1.657 4.019 1.577 1.513 1.929 1.711 1.187]
 [1.303 1.299 1.577 2.373 1.166 1.501 1.366 0.953]
 [1.218 1.158 1.513 1.166 2.052 1.399 1.296 0.944]
 [1.674 1.516 1.929 1.501 1.399 3.289 1.9   1.034]
 [1.561 1.808 1.711 1.366 1.296 1.9   2.759 0.916]
 [0.876 0.761 1.187 0.953 0.944 1.034 0.916 1.577]]


### 📊 Optimización de Portafolio con la Frontera Eficiente (Modelo de Markowitz)

En este análisis se utiliza la librería $PyPortfolioOpt$ para construir un portafolio óptimo basado en la teoría de la media-varianza de Harry Markowitz. El objetivo es determinar cómo distribuir un capital total (por ejemplo, 1,000,000) entre diferentes activos financieros para maximizar el rendimiento ajustado al riesgo.

---

### 🧠 ¿Qué es la frontera eficiente?

La frontera eficiente representa el conjunto de portafolios óptimos que ofrecen la **mayor rentabilidad esperada para un nivel dado de riesgo**, o bien, **el menor riesgo posible para una rentabilidad esperada**. Dentro de esta frontera, uno de los puntos más relevantes es el portafolio de **máxima relación de Sharpe**.

---

### ⚙️ ¿Qué hace la optimización?

1. **Entrada**:
   - Un vector de **retornos promedio esperados** para cada activo.
   - Una **matriz de varianza-covarianza** de los rendimientos.
   - Límites sobre los pesos de cada activo en el portafolio (por ejemplo, mínimo 0%, máximo 50%).

2. **Criterio de optimización**:
   - Maximizar la **relación de Sharpe**, definida como:

     $$
     \text{Sharpe Ratio} = \frac{\mathbb{E}[R_p] - R_f}{\sigma_p}
     $$

     Donde:
     - $ \mathbb{E}[R_p] $ es el retorno esperado del portafolio.
     - $ R_f $ es la tasa libre de riesgo (usualmente se asume 0 si no se especifica).
     - $ \sigma_p$ es la desviación estándar del portafolio (riesgo).

3. **Salida**:
   - Un conjunto de **pesos óptimos** que indican qué proporción del capital debe asignarse a cada activo.
   - Estos pesos pueden convertirse en **valores monetarios absolutos** multiplicándolos por el capital total.

---

### 🎯 ¿Qué representa el resultado?

El resultado es un portafolio teóricamente óptimo que busca **maximizar el rendimiento ajustado al riesgo**, cumpliendo con las restricciones impuestas. Es decir:

- Diversificado (evita sobreexposición a un solo activo).
- Eficiente (aprovecha la correlación entre activos).
- Sintonizado con una estrategia conservadora o moderada, según los límites de peso.

Este enfoque sirve como referencia cuantitativa para tomar decisiones de inversión estructuradas y justificadas en teoría financiera moderna.

---


In [ ]:
# Importa la clase EfficientFrontier de la librería PyPortfolioOpt,
# que permite construir portafolios óptimos bajo diferentes criterios (Sharpe, riesgo mínimo, etc.)
from pypfopt.efficient_frontier import EfficientFrontier

# Crea un optimizador de portafolio usando los retornos medios y la matriz de covarianza
# Se imponen límites a los pesos: cada activo puede tener entre 0% y 50% del portafolio
ef_mean = EfficientFrontier(meanReturns, covReturns, weight_bounds=(0, 0.5))

# Calcula los pesos que maximizan la relación de Sharpe del portafolio
# Esto busca el mejor trade-off entre retorno esperado y riesgo (volatilidad)
raw_weights_mean = ef_mean.max_sharpe()

# Limpia los pesos: elimina valores cercanos a cero y los redondea para interpretación más clara
cleaned_weights_mean = ef_mean.clean_weights()

# Convierte los pesos proporcionales a cantidades monetarias, asumiendo un capital total de $1,000,000
# Se genera un arreglo NumPy con 8 valores, uno por cada acción (orden según los tickers originales)
mvo_weights = np.array([1000000 * cleaned_weights_mean[i] for i in range(8)])

# Visualiza el resultado: cuánto invertir en cada activo para maximizar el Sharpe ratio
mvo_weights

### 📈 Teoría de la Relación de Sharpe

La **relación de Sharpe** es una medida fundamental en finanzas modernas que evalúa el rendimiento ajustado al riesgo de una inversión o portafolio. Fue introducida por el premio Nobel William F. Sharpe.

---

#### 🧠 Definición formal

La relación de Sharpe se define como:

$$
\text{Sharpe Ratio} = \frac{\mathbb{E}[R_p] - R_f}{\sigma_p}
$$

Donde:

- $ \mathbb{E}[R_p] $: retorno esperado del portafolio
- $ R_f $: tasa libre de riesgo (ej. bonos del gobierno)
- $ \sigma_p $: desviación estándar del portafolio (riesgo o volatilidad)

---

#### 🔍 ¿Qué representa?

Esta fórmula responde a:

> *¿Cuánto rendimiento adicional obtengo por cada unidad de riesgo asumido?*

- El **numerador** representa el exceso de retorno sobre una inversión sin riesgo.
- El **denominador** mide la variabilidad de los retornos (riesgo total).

---

#### 📊 Interpretación del Sharpe Ratio

| Sharpe Ratio     | Interpretación                      |
|------------------|-------------------------------------|
| < 1              | Bajo: retorno insuficiente para el riesgo |
| ≈ 1              | Aceptable: riesgo bien compensado   |
| 1.5 – 2          | Bueno: inversión eficiente          |
| > 2              | Excelente: muy buen rendimiento ajustado al riesgo |

Un portafolio con alta volatilidad puede tener un bajo Sharpe si su retorno no compensa el riesgo asumido.

---

#### ⚙️ En contextos prácticos

En muchos experimentos, como en backtesting académico, se asume $ R_f = 0 $, y la fórmula se simplifica:

$$
\text{Sharpe} = \frac{\mathbb{E}[R_p]}{\sigma_p}
$$

Esto permite comparar distintas estrategias bajo las mismas condiciones, aunque en producción es más correcto incluir un $ R_f $ realista.

---

#### 📌 Importancia

- **Evalúa eficiencia**: No basta con ganar más, también hay que hacerlo de forma estable.
- **Compara estrategias**: Normaliza resultados para poder comparar entre diferentes activos o métodos.
- **Base para optimización**: Muchos modelos de optimización (como Markowitz) usan el Sharpe como objetivo.

---

#### 📈 Ejemplo comparativo

| Portafolio | Retorno esperado | Volatilidad | Sharpe |
|------------|------------------|-------------|--------|
| A          | 10%              | 5%          | 2.0    |
| B          | 15%              | 15%         | 1.0    |

Aunque el portafolio B tiene mayor retorno, el portafolio A es más eficiente desde el punto de vista de riesgo-retorno.

---

### ✅ Conclusión

La relación de Sharpe es una métrica clave para seleccionar estrategias de inversión inteligentes y sostenibles. Optimizar un portafolio para maximizar el Sharpe ratio significa buscar el mejor equilibrio entre retorno y riesgo, que es precisamente lo que hacen muchas librerías modernas como `PyPortfolioOpt`.

---


### 💼 Interpretación de los Pesos Óptimos del Portafolio

Tras aplicar una optimización basada en la **maximización de la relación de Sharpe**, se obtiene una propuesta de asignación de capital entre los distintos activos del portafolio. Esta asignación está expresada como una serie de montos recomendados para invertir en cada activo, calculados en función del capital total disponible (por ejemplo, $1,000,000).

---

#### 📊 ¿Qué significa esta asignación?

Cada valor corresponde a la cantidad de dinero que debería destinarse a un activo específico para construir un portafolio eficiente. La posición de cada valor está alineada con el orden original de los activos en los datos de entrada. Los valores positivos indican inversión, mientras que los ceros indican que ese activo no debe incluirse en el portafolio.

---

#### 🧠 ¿Por qué algunos activos reciben una asignación de $0?

Esto puede deberse a varios factores:

- El activo presenta un rendimiento esperado bajo en comparación con su nivel de riesgo.
- Su inclusión no mejora la relación retorno/riesgo del portafolio global.
- El modelo optimizador encuentra que es más eficiente asignar el capital a un subconjunto más reducido de activos.
- Se han aplicado restricciones como límites máximos por activo, que impiden diversificar más ampliamente.

---

#### ⚖️ ¿Qué busca esta optimización?

El objetivo no es maximizar el retorno absoluto, sino encontrar el mejor equilibrio entre retorno y riesgo. Es decir:

- Obtener la **mayor rentabilidad esperada por cada unidad de riesgo** asumido.
- Construir un portafolio **eficiente**, donde ninguna redistribución del capital podría mejorar el rendimiento sin aumentar el riesgo.

---

#### 🎯 Conclusión

Esta asignación óptima representa una estrategia cuantitativa sólida, alineada con la **teoría moderna de portafolio**, que favorece activos con alta eficiencia (retorno ajustado a la volatilidad) y descarta aquellos que no contribuyen positivamente al perfil de riesgo-retorno del portafolio.

---


### Apply the weights to the last price of our stock data

### 🧮 Conversión de pesos monetarios a unidades de activos

Después de obtener los montos óptimos que se deben invertir en cada activo (por ejemplo, mediante optimización de portafolio con máxima relación de Sharpe), es necesario convertir esos montos en **cantidades concretas de acciones** que se pueden comprar.

---

#### 🧠 ¿Por qué hacemos esto?

Los resultados de la optimización nos indican, por ejemplo:

- Invertir \$389,010 en el activo 1  
- Invertir \$278,200 en el activo 4  
- Etc.

Sin embargo, para implementar ese portafolio en un entorno simulado o real, necesitamos saber **cuántas unidades (acciones) de cada activo se deben comprar**, lo cual depende de los precios actuales de mercado.

---

#### ⚙️ ¿Cómo se hace la conversión?

Se sigue esta lógica:

$$
\text{número de acciones} = \frac{\text{dinero asignado}}{\text{precio actual por acción}}
$$

Esto equivale a multiplicar:

$$
\text{acciones} = \text{monto en dólares} \times \left( \frac{1}{\text{precio}} \right)
$$

---

#### 📈 Resultado final

El resultado es un vector donde cada entrada representa **la cantidad de unidades (acciones)** que se pueden comprar de cada activo con base en:

- El capital asignado por el optimizador (`mvo_weights`)
- El precio más reciente de cada activo (`StockData.tail(1)`)

Este vector es esencial para:

- Ejecutar el portafolio en entornos simulados o reales.
- Inicializar correctamente un entorno de Aprendizaje por Refuerzo.
- Evaluar el desempeño futuro del portafolio en función de unidades reales de activos.

---

#### ✅ Conclusión

Esta conversión permite pasar de una asignación teórica de capital a una configuración concreta y operativa de portafolio, alineando la teoría financiera con la práctica computacional.

---


In [ ]:
# Obtiene la última fila del DataFrame de precios (StockData), que contiene los precios de cierre más recientes
# Convierte esa fila en un arreglo NumPy de una dimensión
# Luego, calcula el recíproco (1/precio) de cada precio, para usarlo como factor de conversión
# Esto permite más adelante convertir montos en dólares a número de acciones
LastPrice = np.array([1/p for p in StockData.tail(1).to_numpy()[0]])

# Multiplica los montos asignados a cada activo (mvo_weights) por 1/precio
# Resultado: cuántas acciones se pueden comprar de cada activo con el dinero asignado
Initial_Portfolio = np.multiply(mvo_weights, LastPrice)

# Muestra el portafolio inicial expresado como número de acciones por activo
Initial_Portfolio

### Test our MVO weights on the out-of-sample data

### 📈 Evaluación del Portafolio Optimizado (Media-Varianza)

Una vez obtenida la cantidad de unidades (acciones) que deben comprarse de cada activo en el portafolio optimizado, es necesario evaluar cómo habría evolucionado su valor a lo largo del tiempo. Este análisis se realiza bajo una estrategia de tipo **buy-and-hold**, es decir, manteniendo fija la asignación inicial de activos.

---

#### 🧠 ¿Qué se está haciendo?

1. **`Initial_Portfolio`**  
   Vector que indica cuántas acciones se compraron de cada activo al inicio, según la asignación óptima obtenida mediante media-varianza y los precios más recientes.

2. **`TradeData`**  
   Matriz donde:
   - Cada fila representa un día.
   - Cada columna representa un activo.
   - Cada celda contiene el precio de ese activo en esa fecha.

---

#### 🔢 Multiplicación matricial

Se realiza el producto entre:

- La matriz de precios a lo largo del tiempo: $ T \times N $.
- El vector columna con la cantidad de acciones de cada activo: $ N \times 1 $.

Esto da como resultado un vector columna de dimensión $ T \times 1 $ que representa el valor total del portafolio día a día:

$$
\text{Portfolio}_t = \sum_{i=1}^{N} \text{Precio}_{i,t} \cdot \text{Acciones}_i
$$

---

#### 📊 Resultado

El resultado se almacena en un `DataFrame` llamado `"Mean Var"`, donde cada fila representa el valor del portafolio optimizado en un día específico.

---

#### 🎯 ¿Para qué sirve?

Este resultado permite:

- Visualizar la evolución del portafolio optimizado en el tiempo.
- Comparar su rendimiento contra estrategias alternativas (como agentes de RL o buy-and-hold).
- Calcular métricas clave como:
  - Retorno acumulado,
  - Máximo drawdown,
  - Relación de Sharpe, entre otras.

---


In [ ]:
Portfolio_Assets = TradeData @ Initial_Portfolio
MVO_result = pd.DataFrame(Portfolio_Assets, columns=["Mean Var"])

MVO_result

    
import pickle

with open("MVO_result.pkl", "wb") as f:
    pickle.dump(MVO_result, f)

# DOW JONES Index as Another Baseline

In [ ]:
TRAIN_START_DATE = '2009-01-01'
TRAIN_END_DATE = '2020-07-01'
TRADE_START_DATE = '2020-07-01'
TRADE_END_DATE = '2023-05-01'

df_dji = YahooDownloader(start_date = TRADE_START_DATE,
                     end_date = TRADE_END_DATE,
                     ticker_list = ['^dji']).fetch_data()

### 📘 Teoría: ¿Qué es el Dow Jones Industrial Average (DJI)?

El **Dow Jones Industrial Average (DJI)** es uno de los índices bursátiles más antiguos, conocidos y seguidos del mundo. Fue creado en 1896 por Charles Dow y representa el desempeño de **30 grandes empresas industriales** que cotizan en la bolsa de valores de Nueva York (NYSE) y el Nasdaq.

---

#### 🔎 Características principales:

- **Tipo de índice**: ponderado por precio, lo que significa que las acciones con mayor precio tienen más influencia en el valor del índice.
- **Número de componentes**: 30 compañías líderes de la economía estadounidense.
- **Sectores representados**: incluye empresas de tecnología, salud, finanzas, energía, consumo, entre otros.
- **Representatividad**: se considera un termómetro del mercado estadounidense, especialmente de empresas consolidadas.

---

#### 📈 ¿Por qué se usa como benchmark?

El DJI se utiliza como punto de referencia para evaluar el desempeño de estrategias de inversión y otros portafolios, debido a que:

- Tiene una trayectoria histórica amplia y confiable.
- Es seguido globalmente por analistas, inversores y medios financieros.
- Refleja de manera simplificada la salud del mercado bursátil estadounidense tradicional.

---

#### ⚖️ ¿Por qué normalizarlo a 1,000,000?

Cuando se desea comparar el rendimiento de un portafolio simulado (por ejemplo, generado mediante Aprendizaje por Refuerzo o una optimización de Sharpe) con el índice DJI, es útil llevar ambos a una misma escala.

Al escalar el índice para que comience en **1,000,000**, se logra:

- Facilitar la comparación visual con otros portafolios también escalados a esa cifra.
- Observar claramente cuál tiene mejor desempeño relativo.
- Evaluar si una estrategia supera al mercado (tiene alpha positiva) o se comporta de forma más volátil.

---

### ✅ Conclusión

El DJI sirve como un benchmark de referencia clave. Normalizarlo permite comparar en igualdad de condiciones estrategias activas (como RL o MVO) contra el comportamiento general del mercado.

---


In [ ]:
# Selecciona solo las columnas 'date' y 'close' del índice Dow Jones
df_dji = df_dji[['date','close']]

# Obtiene el precio de cierre del primer día (se usará como referencia para normalizar)
fst_day = df_dji['close'][0]

# Normaliza los precios dividiendo entre el precio del primer día y escalando a 1,000,000
# Esto transforma la serie para que comience en 1,000,000, facilitando la comparación con otros portafolios
# Luego une las fechas con esta serie transformada y establece 'date' como índice
dji = pd.merge(
    df_dji['date'],                                   # columna de fechas
    df_dji['close'].div(fst_day).mul(1000000),        # serie normalizada
    how='outer',
    left_index=True,
    right_index=True
).set_index('date')                                   # establece 'date' como índice temporal



# Si estás usando PPO, toma el DataFrame de resultados (valor del portafolio por día)
# y establece la primera columna (que contiene las fechas) como índice temporal
df_result_ppo = df_account_value_ppo.set_index(df_account_value_ppo.columns[0]) if if_using_ppo else None

# Inicializa un DataFrame vacío para ir construyendo una tabla comparativa de resultados
result = pd.DataFrame()

# Si se está usando PPO, fusiona (merge) el DataFrame vacío con los resultados del PPO
# La fusión se hace con 'outer' join, para conservar todas las fechas posibles
# Se usa el índice (fechas) como clave para alinear los datos
if if_using_ppo:
    result = pd.merge(result, df_result_ppo, how='outer', left_index=True, right_index=True)
    
    
import pickle

with open("df_result_ppo.pkl", "wb") as f:
    pickle.dump(df_result_ppo, f)
    
with open("dji.pkl", "wb") as f:
    pickle.dump(dji, f)

# Merge DataFrames and view Results

In [ ]:
# Crea una lista vacía para los nombres de columnas
col_name = []

# Si estás usando PPO, agrega 'PPO' como nombre de columna para el DataFrame resultante
col_name.append('PPO') if if_using_ppo else None

# Asigna los nombres de columna al DataFrame 'result' (en este punto solo incluye PPO)
result.columns = col_name

# Agrega la serie de resultados del portafolio optimizado por media-varianza (MVO) al DataFrame
# El merge se hace por el índice (fechas), y se conserva toda la información posible (outer join)
result = pd.merge(result, MVO_result, how='outer', left_index=True, right_index=True)

# Agrega la serie del índice Dow Jones (DJI) como benchmark
# Se usa fillna con método 'bfill' para rellenar valores faltantes hacia atrás
result = pd.merge(result, dji, how='outer', left_index=True, right_index=True).fillna(method='bfill')

# Configura el tamaño de la figura para la gráfica
plt.rcParams["figure.figsize"] = (15, 5)

# Crea una nueva figura
plt.figure()

# Dibuja una gráfica con todas las columnas de 'result'
# Muestra cómo evolucionó cada portafolio (PPO, MVO, DJI) en el tiempo
result.plot()

# Consider Joining the Newsletter

https://witty-motivator-1414.ck.page/acb393f729